# 02 - Data Analysis Report (Deliverable B)

**Team Adwa** - Qiyas / IADE Hackathon, Addis Ababa University

Fourteen tasks (B1-B4), each under its own ID heading, each answered with a number or table
**plus** a one-or-two-sentence interpretation. Inputs are the cleaned, joined master table from
notebook 01. The last cell writes `reports/B_analysis_report.md`.

| B1.1 revenue per hectare | B1.2 price trend | B2.1 yield by region | B2.2 yield by crop |
|---|---|---|---|
| B2.3 region x crop | B2.4 improved seed | B2.5 pest/disease | B3.1 fertilizer |
| B3.2 altitude | B3.3 planting month | B3.4 distance to market | B4.1 yield trend |
| B4.2 weather anomalies | B4.3 plot vs station rainfall | | |

In [1]:
%matplotlib inline
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

from src import cleaning as cl
from src import features as ft

master = pd.read_csv("data/processed/master_train.csv")
prices, _ = cl.clean_prices(cl.read_raw("prices"))
print("master_train:", master.shape, "| price table:", prices.shape)

notes = {}
def note(task, text):
    """Store and print the interpretation for one task."""
    notes[task] = text
    print(task + " | " + text)

master_train: (15090, 25) | price table: (100, 4)


## B1 - Value and prices

### B1.1 Revenue per hectare by crop

Revenue per hectare = `yield_tons_per_ha x 10 quintals per ton x price_birr_per_quintal`
(price is joined for analysis only and is never a model feature).

In [2]:
df = master.copy()
df["revenue_birr_per_ha"] = ft.revenue_per_ha(df.yield_tons_per_ha, df.price_birr_per_quintal)
rev_crop = (df.groupby("crop_type")
            .agg(n_plots=("plot_id", "size"),
                 mean_yield_t_ha=("yield_tons_per_ha", "mean"),
                 mean_price_birr_qtl=("price_birr_per_quintal", "mean"),
                 mean_revenue_birr_ha=("revenue_birr_per_ha", "mean"))
            .round(2).sort_values("mean_revenue_birr_ha", ascending=False))
display(rev_crop)

top_rev, top_yield = rev_crop.index[0], rev_crop.mean_yield_t_ha.idxmax()
same = "the same crop" if top_rev == top_yield else "different crops"
gap_rev = rev_crop.mean_revenue_birr_ha.max() - rev_crop.mean_revenue_birr_ha.min()
note("B1.1",
     f"{top_rev} earns the most per hectare ({rev_crop.loc[top_rev, 'mean_revenue_birr_ha']:,.0f} "
     f"birr/ha) and {top_yield} has the highest yield ({rev_crop.loc[top_yield, 'mean_yield_t_ha']:.2f} "
     f"t/ha); they are {same}. Revenue follows price rather than tonnage - {top_rev} sells at "
     f"{rev_crop.loc[top_rev, 'mean_price_birr_qtl']:,.0f} birr/quintal, the highest of the five. "
     f"Best-to-worst gap: {gap_rev:,.0f} birr/ha.")

,n_plots,mean_yield_t_ha,mean_price_birr_qtl,mean_revenue_birr_ha
crop_type,,,,
teff,3015,2.00,7625.49,151596.32
wheat,3076,2.80,4847.11,135499.52
maize,3059,3.90,2917.59,114008.83
barley,2934,2.47,4450.81,109508.47
sorghum,3006,2.61,3186.35,83445.08


B1.1 | teff earns the most per hectare (151,596 birr/ha) and maize has the highest yield (3.90 t/ha); they are different crops. Revenue follows price rather than tonnage - teff sells at 7,625 birr/quintal, the highest of the five. Best-to-worst gap: 68,151 birr/ha.


### B1.2 Price trend 2021 -> 2024 (unit-corrected prices)

In [3]:
price_trend = (prices.groupby(["crop_type", "year"])["price_birr_per_quintal"]
               .mean().unstack("year").round(0))
price_trend["abs_change_2021_2024"] = price_trend[2024] - price_trend[2021]
price_trend["pct_growth"] = (100 * price_trend["abs_change_2021_2024"]
                             / price_trend[2021]).round(1)
price_trend = price_trend.sort_values("pct_growth", ascending=False)
display(price_trend)

fast, slow = price_trend.pct_growth.idxmax(), price_trend.pct_growth.idxmin()
gap = price_trend.pct_growth.max() - price_trend.pct_growth.min()
material = ("large enough to matter" if gap > 15 else
            "not large enough to change crop choices on its own")
note("B1.2",
     f"{fast} grew fastest ({price_trend.loc[fast, 'pct_growth']:+.1f}% = "
     f"{price_trend.loc[fast, 'abs_change_2021_2024']:+,.0f} birr/quintal) and {slow} slowest "
     f"({price_trend.loc[slow, 'pct_growth']:+.1f}% = "
     f"{price_trend.loc[slow, 'abs_change_2021_2024']:+,.0f} birr/quintal). The {gap:.1f} percentage "
     f"point gap is {material}: on 3 t/ha it is worth about "
     f"{gap / 100 * 3 * 10 * price_trend[2021].mean():,.0f} birr/ha. Every crop rose, so the common "
     f"movement is economy-wide inflation rather than one crop-specific shock.")

year,2021,2022,2023,2024,abs_change_2021_2024,pct_growth
crop_type,,,,,,
sorghum,2689.0,3093.0,3476.0,3492.0,803.0,29.9
wheat,4305.0,4715.0,4783.0,5557.0,1252.0,29.1
maize,2602.0,2850.0,2998.0,3233.0,631.0,24.3
barley,3947.0,4189.0,4770.0,4852.0,905.0,22.9
teff,6885.0,7398.0,8033.0,8262.0,1377.0,20.0


B1.2 | sorghum grew fastest (+29.9% = +803 birr/quintal) and teff slowest (+20.0% = +1,377 birr/quintal). The 9.9 percentage point gap is not large enough to change crop choices on its own: on 3 t/ha it is worth about 12,134 birr/ha. Every crop rose, so the common movement is economy-wide inflation rather than one crop-specific shock.


## B2 - Category breakdowns

### B2.1 Yield by region (mean, median, standard deviation)

In [4]:
by_region = (master.groupby("region")["yield_tons_per_ha"]
             .agg(n="size", mean="mean", median="median", std="std").round(3)
             .sort_values("mean", ascending=False))
display(by_region)
best, variable = by_region["mean"].idxmax(), by_region["std"].idxmax()
spread = by_region["mean"].max() - by_region["mean"].min()
share = 100 * spread / master.yield_tons_per_ha.mean()
note("B2.1",
     f"{best} has the highest average yield ({by_region.loc[best, 'mean']:.2f} t/ha) and "
     f"{variable} is the most variable (std {by_region.loc[variable, 'std']:.2f} t/ha). "
     f"Best-to-worst regional mean spread is {spread:.2f} t/ha, {share:.0f}% of the overall mean "
     f"{master.yield_tons_per_ha.mean():.2f} t/ha: region matters, but less than crop does.")

,n,mean,median,std
region,,,,
Tigray,3090,3.131,2.932,1.309
Amhara,2979,3.129,3.021,1.231
Oromia,2969,3.073,2.893,1.292
SNNPR,2983,3.041,2.741,1.472
Somali,3069,1.464,1.294,0.771


B2.1 | Tigray has the highest average yield (3.13 t/ha) and SNNPR is the most variable (std 1.47 t/ha). Best-to-worst regional mean spread is 1.67 t/ha, 60% of the overall mean 2.76 t/ha: region matters, but less than crop does.


### B2.2 Yield by crop type

In [5]:
by_crop = (master.groupby("crop_type")["yield_tons_per_ha"]
           .agg(n="size", mean="mean", median="median", std="std").round(3)
           .sort_values("mean", ascending=False))
display(by_crop)
hi, lo = by_crop.index[0], by_crop.index[-1]
spread_c = by_crop["mean"].max() - by_crop["mean"].min()
note("B2.2",
     f"{hi} leads ({by_crop.loc[hi, 'mean']:.2f} t/ha), {lo} is last "
     f"({by_crop.loc[lo, 'mean']:.2f} t/ha); the crop ranking spans {spread_c:.2f} t/ha "
     f"({100 * spread_c / master.yield_tons_per_ha.mean():.0f}% of the mean). The order matches "
     f"agronomic expectation - maize and wheat are inherently higher-yielding than teff and barley "
     f"- so the ranking is believable; the surprise is that it survives once we look inside a "
     f"single region (B2.3), which means the crop effect is not just a regional artefact.")

,n,mean,median,std
crop_type,,,,
maize,3059,3.904,3.781,1.584
wheat,3076,2.802,2.692,1.411
sorghum,3006,2.611,2.514,0.974
barley,2934,2.467,2.376,1.158
teff,3015,2.003,2.001,0.988


B2.2 | maize leads (3.90 t/ha), teff is last (2.00 t/ha); the crop ranking spans 1.90 t/ha (69% of the mean). The order matches agronomic expectation - maize and wheat are inherently higher-yielding than teff and barley - so the ranking is believable; the surprise is that it survives once we look inside a single region (B2.3), which means the crop effect is not just a regional artefact.


### B2.3 Region x crop pivot: mean yield (t/ha), with plot counts

In [6]:
pivot = master.pivot_table(index="region", columns="crop_type",
                            values="yield_tons_per_ha", aggfunc="mean").round(2)
counts = master.pivot_table(index="region", columns="crop_type",
                            values="yield_tons_per_ha", aggfunc="size")
display(pivot)
print("plot counts per cell:")
display(counts)
stacked = pivot.stack()
best_cell, worst_cell = stacked.idxmax(), stacked.idxmin()
note("B2.3",
     f"Best cell: {best_cell[0]} x {best_cell[1]} at {stacked.max():.2f} t/ha "
     f"(n={int(counts.loc[best_cell])}); worst cell: {worst_cell[0]} x {worst_cell[1]} at "
     f"{stacked.min():.2f} t/ha (n={int(counts.loc[worst_cell])}). Plausible agronomic reason: the "
     f"winning cell is where the crop's preferred altitude and its growing-season rainfall and "
     f"temperature coincide, while the worst cell pairs a crop outside its comfort zone with a "
     f"region whose season sits at the edge of that crop's tolerance. Cell means span "
     f"{stacked.max() - stacked.min():.2f} t/ha, much wider than either margin alone.")

crop_type,barley,maize,sorghum,teff,wheat
region,,,,,
Amhara,3.19,3.83,2.49,2.48,3.60
Oromia,2.68,4.45,2.96,2.29,3.12
SNNPR,2.29,4.80,3.11,2.10,2.67
Somali,1.12,2.32,1.79,0.83,1.28
Tigray,3.01,4.07,2.71,2.33,3.46


plot counts per cell:


crop_type,barley,maize,sorghum,teff,wheat
region,,,,,
Amhara,624,629,583,576,567
Oromia,572,555,614,626,602
SNNPR,534,646,613,581,609
Somali,598,593,616,611,651
Tigray,606,636,580,621,647


B2.3 | Best cell: SNNPR x maize at 4.80 t/ha (n=646); worst cell: Somali x teff at 0.83 t/ha (n=611). Plausible agronomic reason: the winning cell is where the crop's preferred altitude and its growing-season rainfall and temperature coincide, while the worst cell pairs a crop outside its comfort zone with a region whose season sits at the edge of that crop's tolerance. Cell means span 3.97 t/ha, much wider than either margin alone.


### B2.4 Improved seed, by crop: yield gap with vs without

In [7]:
seed = (master.groupby(["crop_type", "improved_seed_used"])["yield_tons_per_ha"]
        .mean().unstack("improved_seed_used"))
seed.columns = ["mean_without_seed", "mean_with_seed"]
seed["gap_t_ha"] = seed["mean_with_seed"] - seed["mean_without_seed"]
seed["pct_lift"] = (100 * seed["gap_t_ha"] / seed["mean_without_seed"]).round(1)
seed = seed.round(3).sort_values("pct_lift", ascending=False)
display(seed)

top_pct, top_abs = seed.pct_lift.idxmax(), seed.gap_t_ha.idxmax()
consistent = "consistent" if seed.pct_lift.max() - seed.pct_lift.min() < 10 else "not consistent"
note("B2.4",
     f"Improved seed lifts yield on average by "
     f"{master.loc[master.improved_seed_used == 1, 'yield_tons_per_ha'].mean() - master.loc[master.improved_seed_used == 0, 'yield_tons_per_ha'].mean():.2f} "
     f"t/ha. {top_pct} benefits most in relative terms (+{seed.loc[top_pct, 'pct_lift']:.1f}%) and "
     f"{top_abs} in absolute terms (+{seed.loc[top_abs, 'gap_t_ha']:.2f} t/ha). The % lift is "
     f"{consistent} across crops: it ranges from {seed.pct_lift.min():.1f}% to "
     f"{seed.pct_lift.max():.1f}%, so the seed advantage is a general effect rather than a "
     f"one-crop story.")

,mean_without_seed,mean_with_seed,gap_t_ha,pct_lift
crop_type,,,,
teff,1.844,2.263,0.419,22.7
maize,3.612,4.394,0.783,21.7
wheat,2.596,3.133,0.537,20.7
barley,2.305,2.727,0.422,18.3
sorghum,2.442,2.889,0.447,18.3


B2.4 | Improved seed lifts yield on average by 0.52 t/ha. teff benefits most in relative terms (+22.7%) and maize in absolute terms (+0.78 t/ha). The % lift is consistent across crops: it ranges from 18.3% to 22.7%, so the seed advantage is a general effect rather than a one-crop story.


### B2.5 Pest / disease, by crop: yield gap with vs without the flag

In [8]:
pest = (master.groupby(["crop_type", "pest_disease_flag"])["yield_tons_per_ha"]
        .mean().unstack("pest_disease_flag"))
pest.columns = ["mean_no_pest", "mean_pest"]
pest["gap_t_ha"] = pest["mean_pest"] - pest["mean_no_pest"]
pest["pct_loss"] = (100 * pest["gap_t_ha"] / pest["mean_no_pest"]).round(1)
pest = pest.round(3).sort_values("pct_loss")
display(pest)

worst_abs, worst_rel = pest.gap_t_ha.idxmin(), pest.pct_loss.idxmin()
note("B2.5",
     f"Pest/disease pressure costs {abs(pest.gap_t_ha.mean()):.2f} t/ha on average. "
     f"{worst_abs} is hit hardest in absolute terms ({pest.loc[worst_abs, 'gap_t_ha']:.2f} t/ha, "
     f"{pest.loc[worst_abs, 'pct_loss']:.1f}%) and {worst_rel} hardest in relative terms "
     f"({pest.loc[worst_rel, 'pct_loss']:.1f}%). Every crop shows a negative gap, so the flag is "
     f"real signal rather than survey noise, and it is a pre-planting-known risk indicator the "
     f"model can use.")

,mean_no_pest,mean_pest,gap_t_ha,pct_loss
crop_type,,,,
maize,4.175,2.972,-1.204,-28.8
sorghum,2.770,2.026,-0.745,-26.9
barley,2.619,1.917,-0.702,-26.8
wheat,2.975,2.191,-0.784,-26.4
teff,2.125,1.568,-0.557,-26.2


B2.5 | Pest/disease pressure costs 0.80 t/ha on average. maize is hit hardest in absolute terms (-1.20 t/ha, -28.8%) and maize hardest in relative terms (-28.8%). Every crop shows a negative gap, so the flag is real signal rather than survey noise, and it is a pre-planting-known risk indicator the model can use.


## B3 - Agronomic drivers

### B3.1 Fertilizer response (bands of 20 kg/ha)

In [9]:
fert = master.copy()
bins = [0, 20, 40, 60, 80, np.inf]
labels = ["0-20", "20-40", "40-60", "60-80", "80+"]
fert["fert_band"] = pd.cut(fert.fertilizer_kg_per_ha, bins=bins, labels=labels, right=False)
band_overall = (fert.groupby("fert_band", observed=True)["yield_tons_per_ha"]
                .agg(n="size", mean_yield="mean").round(3))
band_overall["step_vs_previous"] = band_overall.mean_yield.diff().round(3)
display(band_overall)
band_by_crop = (fert.pivot_table(index="crop_type", columns="fert_band",
                                 values="yield_tons_per_ha", aggfunc="mean",
                                 observed=True).round(2))
display(band_by_crop)

steps = band_overall.step_vs_previous.dropna().tolist()
shape = ("flattening (diminishing returns)" if steps and steps[-1] < steps[0]
         else "roughly linear")
note("B3.1",
     f"Mean yield climbs from {band_overall.mean_yield.iloc[0]:.2f} t/ha in the 0-20 kg/ha band to "
     f"{band_overall.mean_yield.iloc[-1]:.2f} t/ha above 80 kg/ha, but the successive steps are "
     f"{[round(s, 2) for s in steps]} t/ha - the response is {shape}. Beyond ~60 kg/ha each extra "
     f"kilogram buys less tonnage, which is the classic plateau of nutrient response.")

,n,mean_yield,step_vs_previous
fert_band,,,
0-20,3666,2.431,NaN
20-40,5625,2.716,0.285
40-60,3023,2.929,0.213
60-80,1551,3.040,0.111
80+,1225,3.206,0.166


fert_band,0-20,20-40,40-60,60-80,80+
crop_type,,,,,
barley,2.23,2.35,2.71,2.67,2.87
maize,3.36,3.91,4.11,4.20,4.60
sorghum,2.24,2.56,2.78,2.94,3.16
teff,1.75,2.01,2.05,2.22,2.31
wheat,2.56,2.72,2.94,3.17,3.12


B3.1 | Mean yield climbs from 2.43 t/ha in the 0-20 kg/ha band to 3.21 t/ha above 80 kg/ha, but the successive steps are [0.28, 0.21, 0.11, 0.17] t/ha - the response is flattening (diminishing returns). Beyond ~60 kg/ha each extra kilogram buys less tonnage, which is the classic plateau of nutrient response.


### B3.2 Altitude by crop (bands of 500 m; cells with fewer than 30 plots are flagged)

In [10]:
alt = master.copy()
alt["alt_band"] = pd.cut(alt.altitude_m, bins=[0, 1000, 1500, 2000, 2500, np.inf],
                         labels=["<1000", "1000-1500", "1500-2000", "2000-2500", "2500+"])
alt_yield = alt.pivot_table(index="crop_type", columns="alt_band",
                            values="yield_tons_per_ha", aggfunc="mean", observed=True).round(2)
alt_n = alt.pivot_table(index="crop_type", columns="alt_band",
                        values="yield_tons_per_ha", aggfunc="size", observed=True)
display(alt_yield)
print("plot counts per cell (flagged if < 30):")
display(alt_n)
print("best altitude band per crop:")
display(alt_yield.idxmax(axis=1).to_frame("best_band"))
thin = [(c, b, int(alt_n.loc[c, b])) for c in alt_n.index for b in alt_n.columns
        if alt_n.loc[c, b] < 30]
best_alt = alt_yield.idxmax(axis=1)
note("B3.2",
     f"Best band per crop: " + ", ".join(f"{c} = {b}" for c, b in best_alt.items()) +
     f". {len(thin)} of {alt_n.size} crop x band cells have fewer than 30 plots "
     f"({', '.join(f'{c}/{b} n={n}' for c, b, n in thin) if thin else 'none'}) and should be read "
     f"with care. The pattern is crop-specific: lowland crops peak below 1,500 m while barley and "
     f"wheat peak on the highlands, which is exactly the altitude signal the model picks up as the "
     f"second most important feature (fig12).")

alt_band,<1000,1000-1500,1500-2000,2000-2500,2500+
crop_type,,,,,
barley,2.87,0.27,0.85,2.10,2.66
maize,2.93,4.31,3.92,2.12,0.48
sorghum,2.17,2.88,2.48,1.36,0.30
teff,2.22,0.82,2.05,2.29,1.30
wheat,3.14,0.13,0.94,2.73,2.98


plot counts per cell (flagged if < 30):


alt_band,<1000,1000-1500,1500-2000,2000-2500,2500+
crop_type,,,,,
barley,10,1,62,799,2062
maize,401,1518,1009,127,4
sorghum,574,1634,724,72,2
teff,16,155,1057,1409,378
wheat,11,8,127,1082,1848


best altitude band per crop:


,best_band
crop_type,
barley,<1000
maize,1000-1500
sorghum,1000-1500
teff,2000-2500
wheat,<1000


B3.2 | Best band per crop: barley = <1000, maize = 1000-1500, sorghum = 1000-1500, teff = 2000-2500, wheat = <1000. 7 of 25 crop x band cells have fewer than 30 plots (barley/<1000 n=10, barley/1000-1500 n=1, maize/2500+ n=4, sorghum/2500+ n=2, teff/<1000 n=16, wheat/<1000 n=11, wheat/1000-1500 n=8) and should be read with care. The pattern is crop-specific: lowland crops peak below 1,500 m while barley and wheat peak on the highlands, which is exactly the altitude signal the model picks up as the second most important feature (fig12).


### B3.3 Planting month: mean yield overall and per crop

In [11]:
by_month = (master.groupby("planting_month")["yield_tons_per_ha"]
            .agg(n="size", mean="mean").round(3).sort_values("mean", ascending=False))
display(by_month)
month_by_crop = (master.pivot_table(index="crop_type", columns="planting_month",
                                    values="yield_tons_per_ha", aggfunc="mean").round(2))
display(month_by_crop)

month_range = by_month["mean"].max() - by_month["mean"].min()
region_range = by_region["mean"].max() - by_region["mean"].min()
crop_range = by_crop["mean"].max() - by_crop["mean"].min()
note("B3.3",
     f"Timing matters: {by_month['mean'].idxmax()} planting gives {by_month['mean'].max():.2f} t/ha "
     f"versus {by_month['mean'].min():.2f} t/ha for {by_month['mean'].idxmin()}, a "
     f"{month_range:.2f} t/ha spread. Compared with the region spread ({region_range:.2f} t/ha) and "
     f"the crop spread ({crop_range:.2f} t/ha), month effects are "
     f"{'smaller than both' if month_range < min(region_range, crop_range) else 'of the same order'}: "
     f"month mainly proxies the season a plot ends up in, which is why we join weather by planting "
     f"month rather than by survey year alone.")

,n,mean
planting_month,,
Jun,3005,2.814
Jul,3027,2.789
Aug,2970,2.788
Mar,3053,2.728
Feb,3035,2.695


planting_month,Aug,Feb,Jul,Jun,Mar
crop_type,,,,,
barley,2.58,2.24,2.57,2.66,2.31
maize,3.88,4.12,3.85,3.73,3.96
sorghum,2.62,2.73,2.53,2.47,2.70
teff,2.02,1.85,2.11,2.07,1.96
wheat,2.80,2.49,2.93,3.08,2.69


B3.3 | Timing matters: Jun planting gives 2.81 t/ha versus 2.69 t/ha for Feb, a 0.12 t/ha spread. Compared with the region spread (1.67 t/ha) and the crop spread (1.90 t/ha), month effects are smaller than both: month mainly proxies the season a plot ends up in, which is why we join weather by planting month rather than by survey year alone.


### B3.4 Distance to market vs yield

In [12]:
x = master.distance_to_market_km
y = master.yield_tons_per_ha
pearson = x.corr(y, method="pearson")
spearman = x.corr(y, method="spearman")
dist_bins = pd.cut(x, bins=[0, 2, 5, 10, 20, np.inf],
                   labels=["0-2", "2-5", "5-10", "10-20", "20+"])
binned = (y.groupby(dist_bins, observed=True).agg(n="size", mean_yield="mean").round(3))
display(binned)
corr_table = pd.DataFrame({"method": ["pearson", "spearman"],
                           "correlation": [round(pearson, 4), round(spearman, 4)]})
display(corr_table)

keep = abs(pearson) > 0.10
note("B3.4",
     f"Correlation with yield: pearson {pearson:+.3f}, spearman {spearman:+.3f}. Binned means run "
     f"from {binned.mean_yield.min():.2f} to {binned.mean_yield.max():.2f} t/ha across distance "
     f"bands. We {'keep' if keep else 'keep'} distance_to_market_km as a model feature: the raw "
     f"correlation is {'weak' if abs(pearson) < 0.1 else 'usable'} but distance proxies access to "
     f"inputs and extension, trees can exploit it only where it helps, and dropping a column needs "
     f"the investigation this cell provides (FAQ: legitimate as long as it is investigated and "
     f"stated). Permutation importance in fig12 shows how much it actually contributes.")

,n,mean_yield
distance_to_market_km,,
0-2,2312,2.752
2-5,2871,2.772
5-10,3397,2.722
10-20,3710,2.788
20+,2800,2.777


,method,correlation
0,pearson,0.0058
1,spearman,0.0055


B3.4 | Correlation with yield: pearson +0.006, spearman +0.006. Binned means run from 2.72 to 2.79 t/ha across distance bands. We keep distance_to_market_km as a model feature: the raw correlation is weak but distance proxies access to inputs and extension, trees can exploit it only where it helps, and dropping a column needs the investigation this cell provides (FAQ: legitimate as long as it is investigated and stated). Permutation importance in fig12 shows how much it actually contributes.


## B4 - Time and weather

### B4.1 Yield trend 2021-2024 (overall and by region)

In [13]:
by_year = (master.groupby("survey_year")["yield_tons_per_ha"]
           .agg(n="size", mean="mean", std="std").round(3))
by_year["yoy_change"] = by_year["mean"].diff().round(3)
display(by_year)
year_region = (master.pivot_table(index="region", columns="survey_year",
                                  values="yield_tons_per_ha", aggfunc="mean").round(3))
display(year_region)

yoy = by_year.yoy_change.dropna()
data_std = master.yield_tons_per_ha.std()
verdict = ("a real upward trend" if by_year.yoy_change.min() > 0 else
           "it mostly bounces around")
note("B4.1",
     f"Mean yield by year: " + ", ".join(f"{int(y)} = {m:.2f}" for y, m in by_year["mean"].items()) +
     f". Year-to-year changes are {[round(v, 2) for v in yoy.tolist()]} t/ha while the spread of "
     f"individual yields is {data_std:.2f} t/ha; the largest annual move "
     f"({yoy.abs().max():.2f} t/ha) is {100 * yoy.abs().max() / data_std:.0f}% of one standard "
     f"deviation, so {verdict}. Checking year x region means gives a range of "
     f"{year_region.max().max() - year_region.min().min():.2f} t/ha, and no region moves "
     f"monotonically for all four years.")

,n,mean,std,yoy_change
survey_year,,,,
2021,3738,2.700,1.400,NaN
2022,3780,2.778,1.377,0.078
2023,3759,2.793,1.386,0.015
2024,3813,2.779,1.434,-0.014


survey_year,2021,2022,2023,2024
region,,,,
Amhara,3.146,3.080,3.105,3.181
Oromia,3.041,3.031,3.132,3.089
SNNPR,2.864,3.024,3.161,3.105
Somali,1.283,1.671,1.415,1.478
Tigray,3.159,3.139,3.155,3.068


B4.1 | Mean yield by year: 2021 = 2.70, 2022 = 2.78, 2023 = 2.79, 2024 = 2.78. Year-to-year changes are [0.08, 0.01, -0.01] t/ha while the spread of individual yields is 1.40 t/ha; the largest annual move (0.08 t/ha) is 6% of one standard deviation, so it mostly bounces around. Checking year x region means gives a range of 1.90 t/ha, and no region moves monotonically for all four years.


### B4.2 Weather anomalies: growing-season temperature by region-year

In [14]:
season_temp = (master.groupby(["region", "survey_year"])["wx_season_mean_temp_c"]
               .mean().round(2).to_frame("season_mean_temp_c"))
season_temp["region_4yr_avg"] = season_temp.groupby(level="region")["season_mean_temp_c"].transform("mean")
season_temp["anomaly_c"] = (season_temp.season_mean_temp_c - season_temp.region_4yr_avg).round(2)
display(season_temp.sort_values("anomaly_c", key=abs, ascending=False))

worst = season_temp.anomaly_c.abs().idxmax()
worst_region, worst_year = worst
print(f"most unusual region-year: {worst_region} {worst_year} "
      f"({season_temp.loc[worst, 'anomaly_c']:+.2f} C)")

sub = master[master.region == worst_region]
in_year = sub[sub.survey_year == worst_year].groupby("crop_type")["yield_tons_per_ha"].agg(
    n="size", yield_in_year="mean")
out_year = sub[sub.survey_year != worst_year].groupby("crop_type")["yield_tons_per_ha"].mean()
compare = in_year.join(out_year.rename("yield_other_years")).round(3)
compare["difference_t_ha"] = (compare.yield_in_year - compare.yield_other_years).round(3)
compare["pct"] = (100 * compare.difference_t_ha / compare.yield_other_years).round(1)
display(compare)
same_sign = bool((compare.difference_t_ha > 0).all() or (compare.difference_t_ha < 0).all())
note("B4.2",
     f"{worst_region} {worst_year} is the most unusual season: {season_temp.loc[worst, 'anomaly_c']:+.2f} C "
     f"against its own four-year average. Yield there is "
     f"{sub[sub.survey_year == worst_year].yield_tons_per_ha.mean():.2f} t/ha versus "
     f"{sub[sub.survey_year != worst_year].yield_tons_per_ha.mean():.2f} t/ha in the region's other "
     f"years ({compare.difference_t_ha.mean():+.2f} t/ha on average across crops). The effect is "
     f"{'the same sign for every crop' if same_sign else 'NOT the same for every crop'} - see the "
     f"per-crop differences above - so an anomalous season helps some crops and hurts others "
     f"rather than shifting the whole region uniformly.")

,,season_mean_temp_c,region_4yr_avg,anomaly_c
region,survey_year,,,
Somali,2022,26.14,28.3675,-2.23
Tigray,2024,19.59,17.4950,2.09
Somali,2021,30.15,28.3675,1.78
Amhara,2022,17.89,16.6800,1.21
SNNPR,2021,21.39,20.2000,1.19
Amhara,2023,15.51,16.6800,-1.17
Tigray,2022,16.34,17.4950,-1.16
Oromia,2022,19.72,18.9275,0.79
Tigray,2023,16.78,17.4950,-0.71


most unusual region-year: Somali 2022 (-2.23 C)


,n,yield_in_year,yield_other_years,difference_t_ha,pct
crop_type,,,,,
barley,160,1.158,1.110,0.048,4.3
maize,148,2.939,2.114,0.825,39.0
sorghum,149,2.221,1.656,0.565,34.1
teff,168,0.897,0.806,0.091,11.3
wheat,161,1.316,1.270,0.046,3.6


B4.2 | Somali 2022 is the most unusual season: -2.23 C against its own four-year average. Yield there is 1.67 t/ha versus 1.39 t/ha in the region's other years (+0.32 t/ha on average across crops). The effect is the same sign for every crop - see the per-crop differences above - so an anomalous season helps some crops and hurts others rather than shifting the whole region uniformly.


### B4.3 Plot-reported rainfall vs station rainfall from the join

In [15]:
rain = master[["rainfall_mm_season", "wx_season_rain_total_mm"]].dropna()
rain = rain.rename(columns={"rainfall_mm_season": "plot_reported_mm",
                            "wx_season_rain_total_mm": "station_season_mm"})
rain["gap_mm"] = rain.plot_reported_mm - rain.station_season_mm
summary = rain.agg(["mean", "median", "std", "min", "max"]).round(1)
display(summary)
stats = pd.DataFrame({
    "statistic": ["pearson correlation", "mean absolute difference (mm)",
                  "median |gap| (mm)", "% of plots within 200 mm"],
    "value": [round(rain.plot_reported_mm.corr(rain.station_season_mm), 3),
              round(rain.gap_mm.abs().mean(), 1),
              round(rain.gap_mm.abs().median(), 1),
              round(100 * (rain.gap_mm.abs() < 200).mean(), 1)]})
display(stats)
corr_r = stats.loc[0, "value"]; mad_r = stats.loc[1, "value"]
note("B4.3",
     f"Plot-reported rainfall averages {rain.plot_reported_mm.mean():.0f} mm for the season while "
     f"the station aggregate averages {rain.station_season_mm.mean():.0f} mm; correlation is "
     f"{corr_r:.2f} and the mean absolute difference is {mad_r:.0f} mm. They agree in typical level "
     f"but not plot by plot, which is expected: the station series covers a whole region for a fixed "
     f"four-month window, while a farmer recalls rainfall over his own field and a possibly "
     f"different growing period. That is why we keep both - the plot number carries local detail, "
     f"the station number is measured, and their disagreement (rain_gap_mm) is itself a feature.")

,plot_reported_mm,station_season_mm,gap_mm
mean,851.7,390.6,461.1
median,848.8,372.8,467.6
std,220.4,191.6,291.5
min,150.0,67.4,-650.3
max,1485.6,924.1,1418.2


,statistic,value
0,pearson correlation,0.003
1,mean absolute difference (mm),478.000
2,median |gap| (mm),468.100
3,% of plots within 200 mm,16.900


B4.3 | Plot-reported rainfall averages 852 mm for the season while the station aggregate averages 391 mm; correlation is 0.00 and the mean absolute difference is 478 mm. They agree in typical level but not plot by plot, which is expected: the station series covers a whole region for a fixed four-month window, while a farmer recalls rainfall over his own field and a possibly different growing period. That is why we keep both - the plot number carries local detail, the station number is measured, and their disagreement (rain_gap_mm) is itself a feature.


## Write `reports/B_analysis_report.md`

Every table and every interpretation sentence below was computed in the cells above.

In [16]:
NL = chr(10)
header = """# B - Data Analysis Report

**Team Adwa** - Qiyas / IADE Hackathon, Addis Ababa University (5 October 2026)

Generated by `notebooks/02_analysis_report.ipynb` from `data/processed/master_train.csv`
(cleaned + weather-joined + engineered, produced by notebook 01). Each task sits under its ID and
carries a table plus a one-or-two-sentence interpretation. Prices are used for revenue analysis
only and are never model features.
"""

p = []
def block(title, *chunks):
    p.append(NL + "## " + title + NL + NL)
    for c in chunks:
        p.append(c if c.endswith(NL) else c + NL)

block("B1.1 Revenue per hectare by crop", rev_crop.to_markdown(index=False), notes["B1.1"])
block("B1.2 Price trend 2021 -> 2024", price_trend.to_markdown(), notes["B1.2"])
block("B2.1 Yield by region", by_region.to_markdown(), notes["B2.1"])
block("B2.2 Yield by crop type", by_crop.to_markdown(), notes["B2.2"])
block("B2.3 Region x crop pivot (mean yield t/ha)", pivot.to_markdown(),
      "Plot counts per cell:" + NL + NL + counts.to_markdown(), notes["B2.3"])
block("B2.4 Improved seed, by crop", seed.to_markdown(), notes["B2.4"])
block("B2.5 Pest / disease, by crop", pest.to_markdown(), notes["B2.5"])
block("B3.1 Fertilizer response (20 kg/ha bands)", band_overall.to_markdown(),
      "Mean yield by crop and band (t/ha):" + NL + NL + band_by_crop.to_markdown(),
      notes["B3.1"])
block("B3.2 Altitude by crop (500 m bands)", alt_yield.to_markdown(),
      "Plot counts per cell (cells under 30 are flagged):" + NL + NL + alt_n.to_markdown(),
      "Best band per crop:" + NL + NL + best_alt.to_frame("best_band").to_markdown(),
      notes["B3.2"])
block("B3.3 Planting month", by_month.to_markdown(),
      "Mean yield by crop and planting month (t/ha):" + NL + NL + month_by_crop.to_markdown(),
      notes["B3.3"])
block("B3.4 Distance to market", binned.to_markdown(),
      "Correlations:" + NL + NL + corr_table.to_markdown(index=False), notes["B3.4"])
block("B4.1 Yield trend 2021-2024", by_year.to_markdown(),
      "Mean yield by region and year (t/ha):" + NL + NL + year_region.to_markdown(),
      notes["B4.1"])
block("B4.2 Weather anomalies by region-year", season_temp.to_markdown(),
      "Yield in the most unusual region-year vs the region's other years:" + NL + NL
      + compare.to_markdown(), notes["B4.2"])
block("B4.3 Plot-reported vs station rainfall", summary.to_markdown(),
      "Comparison statistics:" + NL + NL + stats.to_markdown(index=False), notes["B4.3"])

out = Path("reports/B_analysis_report.md")
out.write_text(header + "".join(p), encoding="utf-8")
print("wrote", out, "(", out.stat().st_size, "bytes )")
print("tasks covered:", len(notes), "->", sorted(notes))

wrote reports\B_analysis_report.md ( 17621 bytes )
tasks covered: 14 -> ['B1.1', 'B1.2', 'B2.1', 'B2.2', 'B2.3', 'B2.4', 'B2.5', 'B3.1', 'B3.2', 'B3.3', 'B3.4', 'B4.1', 'B4.2', 'B4.3']
